# Fine-tune path spike: LoRA on a free Colab T4

Runs the same `finetune/train_lora.py` as the Mac proof, on the synthetic spike set only.
Nothing to upload and no sign-in is needed: the cells clone the public repo at a pinned commit,
build the training data themselves and download the model from Hugging Face.

Before running: Runtime > Change runtime type > T4 GPU. Then Runtime > Run all.
The adapter comes back as `adapter.zip`; the merge and GGUF conversion run on the Mac
(`finetune/convert.md`). The T4 has no bf16, so the script uses fp16 there.

In [ ]:
COMMIT = "a10f9ab1742cc2d63361053f1ebd3e8e5c2f2c7c"  # the commit of the repo that holds finetune/
MODEL_REVISION = "89644892e4d85e24eaac8bacfd4f463576704203"  # Qwen/Qwen3-VL-2B-Instruct
!git clone -q https://github.com/Varun-Kambali/household-docs-vault repo
%cd repo
!git checkout -q $COMMIT
!git log --oneline -1

In [ ]:
!pip install -q "transformers>=4.57" peft accelerate pypdfium2 pydantic psutil
%env PYTHONPATH=src:synth:.

In [ ]:
import torch
from huggingface_hub import snapshot_download

print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())
snapshot_download(
    "Qwen/Qwen3-VL-2B-Instruct", revision=MODEL_REVISION, local_dir="finetune/out/base"
)

In [ ]:
!python -m finetune.stages run prepare -- python -m finetune.prepare
!wc -l finetune/data/*.jsonl

In [ ]:
!python -m finetune.train_lora train --model finetune/out/base \
    --steps 20 --rank 8 --max-pixels 786432
!cat finetune/out/stages/train.json

In [ ]:
!cd finetune/out && zip -qr adapter.zip adapter
from google.colab import files

files.download("finetune/out/adapter.zip")
files.download("finetune/out/stages/train.json")

Copy `adapter.zip` into `finetune/out/` on the Mac and unzip it there, then follow
`finetune/convert.md` from the merge step. Keep `train.json` as the Colab run's record.